In [1]:
%pip install azure-eventhub

StatementMeta(, 8694632b-e55e-4202-b26b-5bbcf152ae9a, 7, Finished, Available, Finished, True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 5.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Not uninstalling typing-extensions at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-ec3b038e-d0cc-48be-84b1-446ce92b65b5
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 2.0.0 requires sentencepiece, which is not installed.
sentence-transformers 2.0.0 requires torchvision, which is not installed.
dash 2.14.0 requires Flask<2.3.0,>=1.0.4, but you have flask 3.0.0 which is incompatible.
dash 2.14.0 requires Werkzeug<2.3.0, but you have

In [ ]:
import time
import uuid
import json
import random
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData


# -------------------------------------
# FABRIC EVENTSTREAM CONNECTION
# -------------------------------------
# Get this from your Event Stream's Custom App source → Event Hub tab → SAS Key Authentication
EVENT_HUB_CONNECTION_STRING = "xxx"

# -------------------------------------
# ENERGY SOURCE TYPES
# -------------------------------------
ENERGY_SOURCES = [
    {"type": "Solar",        "unit": "kWh", "base_capacity": 500,  "variance": 0.4},
    {"type": "Wind",         "unit": "kWh", "base_capacity": 800,  "variance": 0.6},
    {"type": "Marine",       "unit": "kWh", "base_capacity": 300,  "variance": 0.3},
    {"type": "Hydroelectric","unit": "kWh", "base_capacity": 1000, "variance": 0.2},
]

# Anomaly tracking - triggers every hour for SUB_007
ANOMALY_SUBSTATION = "SUB_007"
last_anomaly_hour = -1


# -------------------------------------
# ENERGY SUBSTATION SIMULATOR CLASS
# -------------------------------------
class EnergySubstationSimulator:

    def __init__(self, substation_id: str, substation_name: str, energy_type: str, producer_client):
        self.substation_id   = substation_id
        self.substation_name = substation_name
        self.energy_type     = energy_type
        self.producer_client = producer_client
        source_config        = next((s for s in ENERGY_SOURCES if s["type"] == energy_type), ENERGY_SOURCES[0])
        self.base_capacity   = source_config["base_capacity"]
        self.variance        = source_config["variance"]

    def generate_energy_reading(self):
        global last_anomaly_hour
        hour = datetime.now().hour
        if self.energy_type == "Solar":
            time_factor = max(0, (1 - abs(hour - 13) / 8)) if 6 <= hour <= 20 else 0
        elif self.energy_type == "Wind":
            time_factor = 0.7 + 0.3 * random.random()
        elif self.energy_type == "Marine":
            time_factor = 0.5 + 0.5 * abs((hour % 6) - 3) / 3
        else:
            time_factor = 0.9 + 0.1 * random.random()
        generated_kwh = round(self.base_capacity * time_factor * (1 + random.uniform(-self.variance, self.variance)), 2)
        generated_kwh = max(0, generated_kwh)
        is_anomaly = False
        if self.substation_id == ANOMALY_SUBSTATION and last_anomaly_hour != hour:
            last_anomaly_hour = hour
            is_anomaly        = True
            anomaly_type      = random.choice(["spike", "drop"])
            if anomaly_type == "spike":
                generated_kwh = generated_kwh * random.uniform(2.5, 4.0)
                print(f"⚠️ ANOMALY TRIGGERED: {self.substation_id} - POWER SPIKE")
            else:
                generated_kwh = generated_kwh * random.uniform(0.05, 0.15)
                print(f"⚠️ ANOMALY TRIGGERED: {self.substation_id} - POWER DROP")
            generated_kwh = round(generated_kwh, 2)
        efficiency   = round(random.uniform(0.85, 0.98), 3)
        ambient_temp = round(random.uniform(15, 35), 1)
        return {
            "reading_id":            str(uuid.uuid4()),
            "timestamp":             datetime.now().isoformat(),
            "substation_id":         self.substation_id,
            "substation_name":       self.substation_name,
            "energy_type":           self.energy_type,
            "generated_kwh":         generated_kwh,
            "efficiency_percent":    round(efficiency * 100, 1),
            "ambient_temperature_c": ambient_temp,
            "status":                "operational" if generated_kwh > 0 else "idle",
            "grid_voltage_kv":       round(random.uniform(130, 145), 1),
            "is_anomaly":            is_anomaly,
        }

    def send_to_fabric(self, data):
        try:
            event_data_batch = self.producer_client.create_batch()
            event_data_batch.add(EventData(json.dumps(data)))
            self.producer_client.send_batch(event_data_batch)
            status = "🚨 ANOMALY" if data.get("is_anomaly") else "✔"
            print(f"{status} Energy reading sent to Fabric Event Stream")
        except Exception as e:
            print(f"❌ Failed to send event: {e}")

    def process_reading(self):
        reading = self.generate_energy_reading()
        print(json.dumps(reading, indent=2))
        self.send_to_fabric(reading)


# -------------------------------------
# MAIN LOOP – REALTIME SIMULATION
# -------------------------------------
def main():
    try:
        producer = EventHubProducerClient.from_connection_string(conn_str=EVENT_HUB_CONNECTION_STRING)
        print("✅ Connected to Fabric Event Stream")
    except Exception as e:
        print(f"❌ Failed to connect to Event Stream: {e}")
        return

    SUBSTATION_LIST = [
        EnergySubstationSimulator("SUB_001", "Madrid Solar Park",        "Solar",         producer),
        EnergySubstationSimulator("SUB_002", "Barcelona Wind Farm",      "Wind",          producer),
        EnergySubstationSimulator("SUB_003", "Valencia Marine Station",  "Marine",        producer),
        EnergySubstationSimulator("SUB_004", "Zaragoza Solar Array",     "Solar",         producer),
        EnergySubstationSimulator("SUB_005", "Seville Hydro Plant",      "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_006", "Bilbao Wind Turbines",     "Wind",          producer),
        EnergySubstationSimulator("SUB_007", "Almeria Solar Complex",    "Solar",         producer),  # ANOMALY
        EnergySubstationSimulator("SUB_008", "Galicia Marine Array",     "Marine",        producer),
        EnergySubstationSimulator("SUB_009", "Malaga Solar Farm",        "Solar",         producer),
        EnergySubstationSimulator("SUB_010", "Murcia Wind Park",         "Wind",          producer),
        EnergySubstationSimulator("SUB_011", "Toledo Hydro Station",     "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_012", "Granada Solar Plant",      "Solar",         producer),
        EnergySubstationSimulator("SUB_013", "Asturias Wind Complex",    "Wind",          producer),
        EnergySubstationSimulator("SUB_014", "Tarragona Marine Station", "Marine",        producer),
        EnergySubstationSimulator("SUB_015", "Salamanca Solar Array",    "Solar",         producer),
        EnergySubstationSimulator("SUB_016", "Navarra Wind Farm",        "Wind",          producer),
        EnergySubstationSimulator("SUB_017", "Extremadura Hydro Plant",  "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_018", "Cantabria Marine Array",   "Marine",        producer),
        EnergySubstationSimulator("SUB_019", "Cordoba Solar Complex",    "Solar",         producer),
        EnergySubstationSimulator("SUB_020", "Leon Wind Turbines",       "Wind",          producer),
    ]

    print("🚀 Real-Time Energy Generation → Fabric Eventstream Simulator Started")
    print(f"⚠️  Anomaly Detection: {ANOMALY_SUBSTATION} (Almeria Solar Complex) will have anomalies every hour")
    print("-" * 71)

    try:
        while True:
            substation = random.choice(SUBSTATION_LIST)
            substation.process_reading()
            time.sleep(random.uniform(3, 10))
    except KeyboardInterrupt:
        print("\n⏹ Stopping simulator...")
    finally:
        producer.close()
        print("✅ Connection closed")


if __name__ == "__main__":
    main()

StatementMeta(, 8694632b-e55e-4202-b26b-5bbcf152ae9a, 9, Submitted, Running, Running, True)

✅ Connected to Fabric Event Stream
🚀 Real-Time Energy Generation → Fabric Eventstream Simulator Started
⚠️  Anomaly Detection: SUB_007 (Almeria Solar Complex) will have anomalies every hour
-----------------------------------------------------------------------
{
  "reading_id": "4306ce4d-6b9e-4c29-b8c2-90c27099cc66",
  "timestamp": "2026-03-02T11:43:21.989076",
  "substation_id": "SUB_004",
  "substation_name": "Zaragoza Solar Array",
  "energy_type": "Solar",
  "generated_kwh": 258.07,
  "efficiency_percent": 94.5,
  "ambient_temperature_c": 19.0,
  "status": "operational",
  "grid_voltage_kv": 141.3,
  "is_anomaly": false
}
✔ Energy reading sent to Fabric Event Stream
{
  "reading_id": "e8641b3a-2d52-4da0-95eb-f3caf445aa47",
  "timestamp": "2026-03-02T11:43:30.591211",
  "substation_id": "SUB_012",
  "substation_name": "Granada Solar Plant",
  "energy_type": "Solar",
  "generated_kwh": 306.21,
  "efficiency_percent": 88.9,
  "ambient_temperature_c": 30.5,
  "status": "operational",


In [2]:
import pandas as pd

substations_data = [
    {"substation_id":"SUB_001","substation_name":"Madrid Solar Park","energy_type":"Solar","city":"Madrid","region":"Community of Madrid","country":"Spain","postal_code":"28220","address":"Carretera de Colmenar Viejo, km 15","latitude":40.5423,"longitude":-3.6345,"installation_date":"2020-05-15","capacity_mw":50.0,"number_of_panels_turbines":150000,"area_hectares":85.0,"average_annual_output_gwh":75.5},
    {"substation_id":"SUB_002","substation_name":"Barcelona Wind Farm","energy_type":"Wind","city":"Barcelona","region":"Catalonia","country":"Spain","postal_code":"08830","address":"Collserola Natural Park","latitude":41.4231,"longitude":2.1123,"installation_date":"2019-03-22","capacity_mw":80.0,"number_of_panels_turbines":40,"area_hectares":120.0,"average_annual_output_gwh":168.0},
    {"substation_id":"SUB_003","substation_name":"Valencia Marine Station","energy_type":"Marine","city":"Valencia","region":"Valencian Community","country":"Spain","postal_code":"46024","address":"Port of Valencia - Marine Energy Complex","latitude":39.4561,"longitude":-0.3318,"installation_date":"2021-09-10","capacity_mw":30.0,"number_of_panels_turbines":15,"area_hectares":5.0,"average_annual_output_gwh":52.6},
    {"substation_id":"SUB_004","substation_name":"Zaragoza Solar Array","energy_type":"Solar","city":"Zaragoza","region":"Aragon","country":"Spain","postal_code":"50720","address":"Polígono Industrial El Pradillo","latitude":41.7342,"longitude":-0.9534,"installation_date":"2018-11-28","capacity_mw":65.0,"number_of_panels_turbines":195000,"area_hectares":110.0,"average_annual_output_gwh":98.8},
    {"substation_id":"SUB_005","substation_name":"Seville Hydro Plant","energy_type":"Hydroelectric","city":"Seville","region":"Andalusia","country":"Spain","postal_code":"41920","address":"Embalse de la Minilla","latitude":37.5443,"longitude":-5.7234,"installation_date":"2017-06-12","capacity_mw":100.0,"number_of_panels_turbines":4,"area_hectares":250.0,"average_annual_output_gwh":245.0},
    {"substation_id":"SUB_006","substation_name":"Bilbao Wind Turbines","energy_type":"Wind","city":"Bilbao","region":"Basque Country","country":"Spain","postal_code":"48920","address":"Monte Oiz Wind Complex","latitude":43.3213,"longitude":-2.6534,"installation_date":"2019-08-05","capacity_mw":72.0,"number_of_panels_turbines":36,"area_hectares":95.0,"average_annual_output_gwh":151.2},
    {"substation_id":"SUB_007","substation_name":"Almeria Solar Complex","energy_type":"Solar","city":"Almeria","region":"Andalusia","country":"Spain","postal_code":"04720","address":"Tabernas Desert Solar Park","latitude":37.0567,"longitude":-2.3521,"installation_date":"2020-01-20","capacity_mw":90.0,"number_of_panels_turbines":270000,"area_hectares":150.0,"average_annual_output_gwh":145.8},
    {"substation_id":"SUB_008","substation_name":"Galicia Marine Array","energy_type":"Marine","city":"A Coruña","region":"Galicia","country":"Spain","postal_code":"15002","address":"Atlantic Coast Wave Energy Site","latitude":43.3623,"longitude":-8.4115,"installation_date":"2022-03-15","capacity_mw":25.0,"number_of_panels_turbines":12,"area_hectares":3.5,"average_annual_output_gwh":43.8},
    {"substation_id":"SUB_009","substation_name":"Malaga Solar Farm","energy_type":"Solar","city":"Malaga","region":"Andalusia","country":"Spain","postal_code":"29590","address":"Parque Tecnológico de Andalucía","latitude":36.6885,"longitude":-4.4936,"installation_date":"2021-07-18","capacity_mw":55.0,"number_of_panels_turbines":165000,"area_hectares":92.0,"average_annual_output_gwh":88.2},
    {"substation_id":"SUB_010","substation_name":"Murcia Wind Park","energy_type":"Wind","city":"Murcia","region":"Region of Murcia","country":"Spain","postal_code":"30530","address":"Sierra de la Pila","latitude":38.1623,"longitude":-1.2345,"installation_date":"2020-11-03","capacity_mw":68.0,"number_of_panels_turbines":34,"area_hectares":105.0,"average_annual_output_gwh":142.8},
    {"substation_id":"SUB_011","substation_name":"Toledo Hydro Station","energy_type":"Hydroelectric","city":"Toledo","region":"Castilla-La Mancha","country":"Spain","postal_code":"45600","address":"Embalse de Castrejón","latitude":39.7834,"longitude":-4.3521,"installation_date":"2018-04-25","capacity_mw":85.0,"number_of_panels_turbines":3,"area_hectares":180.0,"average_annual_output_gwh":208.5},
    {"substation_id":"SUB_012","substation_name":"Granada Solar Plant","energy_type":"Solar","city":"Granada","region":"Andalusia","country":"Spain","postal_code":"18320","address":"Valle de Lecrín Solar Complex","latitude":36.9234,"longitude":-3.5678,"installation_date":"2019-09-12","capacity_mw":48.0,"number_of_panels_turbines":144000,"area_hectares":80.0,"average_annual_output_gwh":76.8},
    {"substation_id":"SUB_013","substation_name":"Asturias Wind Complex","energy_type":"Wind","city":"Oviedo","region":"Asturias","country":"Spain","postal_code":"33840","address":"Cordillera Cantábrica Wind Farm","latitude":43.2456,"longitude":-5.9123,"installation_date":"2020-02-28","capacity_mw":76.0,"number_of_panels_turbines":38,"area_hectares":115.0,"average_annual_output_gwh":159.6},
    {"substation_id":"SUB_014","substation_name":"Tarragona Marine Station","energy_type":"Marine","city":"Tarragona","region":"Catalonia","country":"Spain","postal_code":"43004","address":"Port of Tarragona - Wave Energy Hub","latitude":41.1167,"longitude":1.2444,"installation_date":"2022-06-20","capacity_mw":28.0,"number_of_panels_turbines":14,"area_hectares":4.2,"average_annual_output_gwh":49.0},
    {"substation_id":"SUB_015","substation_name":"Salamanca Solar Array","energy_type":"Solar","city":"Salamanca","region":"Castilla y León","country":"Spain","postal_code":"37185","address":"Villamayor Solar Park","latitude":40.9943,"longitude":-5.6789,"installation_date":"2021-03-10","capacity_mw":52.0,"number_of_panels_turbines":156000,"area_hectares":88.0,"average_annual_output_gwh":83.2},
    {"substation_id":"SUB_016","substation_name":"Navarra Wind Farm","energy_type":"Wind","city":"Pamplona","region":"Navarre","country":"Spain","postal_code":"31191","address":"Bardenas Reales Wind Complex","latitude":42.3667,"longitude":-1.5234,"installation_date":"2019-12-15","capacity_mw":82.0,"number_of_panels_turbines":41,"area_hectares":125.0,"average_annual_output_gwh":172.2},
    {"substation_id":"SUB_017","substation_name":"Extremadura Hydro Plant","energy_type":"Hydroelectric","city":"Cáceres","region":"Extremadura","country":"Spain","postal_code":"10600","address":"Embalse de Alcántara","latitude":39.7223,"longitude":-6.5234,"installation_date":"2018-08-30","capacity_mw":95.0,"number_of_panels_turbines":4,"area_hectares":220.0,"average_annual_output_gwh":233.0},
    {"substation_id":"SUB_018","substation_name":"Cantabria Marine Array","energy_type":"Marine","city":"Santander","region":"Cantabria","country":"Spain","postal_code":"39004","address":"Bay of Biscay Energy Station","latitude":43.4623,"longitude":-3.8099,"installation_date":"2021-11-08","capacity_mw":32.0,"number_of_panels_turbines":16,"area_hectares":5.5,"average_annual_output_gwh":56.0},
    {"substation_id":"SUB_019","substation_name":"Cordoba Solar Complex","energy_type":"Solar","city":"Cordoba","region":"Andalusia","country":"Spain","postal_code":"14440","address":"Campiña Solar Park","latitude":37.8845,"longitude":-4.7796,"installation_date":"2020-09-05","capacity_mw":58.0,"number_of_panels_turbines":174000,"area_hectares":97.0,"average_annual_output_gwh":92.8},
    {"substation_id":"SUB_020","substation_name":"Leon Wind Turbines","energy_type":"Wind","city":"León","region":"Castilla y León","country":"Spain","postal_code":"24750","address":"Montaña Leonesa Wind Park","latitude":42.8234,"longitude":-5.5678,"installation_date":"2020-05-22","capacity_mw":70.0,"number_of_panels_turbines":35,"area_hectares":110.0,"average_annual_output_gwh":147.0},
]

df_substations = pd.DataFrame(substations_data)
print("=" * 120)
print("ENERGY SUBSTATION DIMENSION TABLE - 20 SUBSTATIONS")
print("=" * 120)
print(df_substations.to_string(index=False))
print(f"\nTotal Substations: {len(df_substations)}")
print(f"Total Capacity: {df_substations['capacity_mw'].sum():.1f} MW")
print(f"Total Annual Output: {df_substations['average_annual_output_gwh'].sum():.1f} GWh")

df_substations.to_csv('substation_dimension.csv', index=False)
print("\n✅ Exported to 'substation_dimension.csv'")

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    spark_df = spark.createDataFrame(df_substations)
    spark_df.write.format("delta").mode("overwrite").saveAsTable("dim_substation")
    spark_df.write.mode("overwrite").parquet("Files/dimensions/dim_substation.parquet")
    print("✅ Saved to Lakehouse as Delta table: 'dim_substation'")
    print("✅ Saved to Lakehouse Files: 'Files/dimensions/dim_substation.parquet'")
except Exception as e:
    print(f"⚠️ Note: To save to Lakehouse, run this in a Fabric notebook attached to a Lakehouse")
    print(f"   Error: {e}")

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 9, Finished, Available, Finished, False)

ENERGY SUBSTATION DIMENSION TABLE - 20 SUBSTATIONS
substation_id          substation_name   energy_type      city              region country postal_code                                  address  latitude  longitude installation_date  capacity_mw  number_of_panels_turbines  area_hectares  average_annual_output_gwh
      SUB_001        Madrid Solar Park         Solar    Madrid Community of Madrid   Spain       28220       Carretera de Colmenar Viejo, km 15   40.5423    -3.6345        2020-05-15         50.0                     150000           85.0                       75.5
      SUB_002      Barcelona Wind Farm          Wind Barcelona           Catalonia   Spain       08830                  Collserola Natural Park   41.4231     2.1123        2019-03-22         80.0                         40          120.0                      168.0
      SUB_003  Valencia Marine Station        Marine  Valencia Valencian Community   Spain       46024 Port of Valencia - Marine Energy Complex   39.4561 

## 🏗️ Enrichment Dimension Tables
The cells below define **static reference tables** that enrich the real-time energy stream  
and provide the entities / relationships needed to build a **knowledge graph / ontology**.

| Table | Ontology Role | Key FK Relationships |
|---|---|---|
| `dim_maintenance` | Event | → dim_substation, dim_technician, dim_equipment |
| `dim_technician` | Person / Agent | → dim_maintenance |
| `dim_weather_station` | Observation context | → dim_substation (nearest) |
| `dim_energy_tariff` | Economic rule | → dim_substation (by energy_type) |
| `dim_equipment` | Physical asset | → dim_substation |
| `dim_regulation` | Legal constraint | → bridge_substation_regulation |
| `dim_grid_operator` | Organisation | → dim_substation (grid connection) |
| `dim_alert_type` | Classification taxonomy | → fact_energy_reading (is_anomaly) |
| `bridge_substation_regulation` | M:M compliance | → dim_substation, dim_regulation |

In [3]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_MAINTENANCE  –  Maintenance & Inspection Log
# Tracks every scheduled / corrective / inspection event per substation.
# Ontology roles: Maintenance EVENT linked to Substation, Technician, Equipment
# ─────────────────────────────────────────────────────────────────────────────
maintenance_data = [
    {"maintenance_id":"MNT_001","substation_id":"SUB_001","maintenance_type":"Preventive","component":"Solar Panels","scheduled_date":"2024-02-10","completed_date":"2024-02-10","technician_id":"TECH_03","duration_hours":6.0,"cost_eur":3200.0,"status":"Completed","description":"Annual panel cleaning and efficiency check","next_maintenance_date":"2025-02-10","downtime_hours":2.0},
    {"maintenance_id":"MNT_002","substation_id":"SUB_002","maintenance_type":"Corrective","component":"Gearbox","scheduled_date":"2024-03-05","completed_date":"2024-03-07","technician_id":"TECH_07","duration_hours":18.0,"cost_eur":12500.0,"status":"Completed","description":"Gearbox bearing replacement after vibration anomaly","next_maintenance_date":"2025-03-05","downtime_hours":18.0},
    {"maintenance_id":"MNT_003","substation_id":"SUB_003","maintenance_type":"Preventive","component":"Tidal Converter","scheduled_date":"2024-01-20","completed_date":"2024-01-21","technician_id":"TECH_12","duration_hours":10.0,"cost_eur":8900.0,"status":"Completed","description":"Marine biofouling removal and seal inspection","next_maintenance_date":"2024-07-20","downtime_hours":10.0},
    {"maintenance_id":"MNT_004","substation_id":"SUB_005","maintenance_type":"Preventive","component":"Turbine Blades","scheduled_date":"2024-04-15","completed_date":"2024-04-15","technician_id":"TECH_02","duration_hours":8.0,"cost_eur":5500.0,"status":"Completed","description":"Hydro turbine blade inspection and cavitation check","next_maintenance_date":"2025-04-15","downtime_hours":3.0},
    {"maintenance_id":"MNT_005","substation_id":"SUB_007","maintenance_type":"Corrective","component":"Inverter","scheduled_date":"2024-05-01","completed_date":"2024-05-02","technician_id":"TECH_05","duration_hours":14.0,"cost_eur":9800.0,"status":"Completed","description":"Inverter replacement following voltage spike anomaly","next_maintenance_date":"2025-05-01","downtime_hours":14.0},
    {"maintenance_id":"MNT_006","substation_id":"SUB_009","maintenance_type":"Preventive","component":"Solar Panels","scheduled_date":"2024-06-12","completed_date":"2024-06-12","technician_id":"TECH_08","duration_hours":5.0,"cost_eur":2800.0,"status":"Completed","description":"Panel realignment and junction box inspection","next_maintenance_date":"2025-06-12","downtime_hours":1.5},
    {"maintenance_id":"MNT_007","substation_id":"SUB_011","maintenance_type":"Preventive","component":"Generator","scheduled_date":"2024-07-08","completed_date":"2024-07-09","technician_id":"TECH_01","duration_hours":12.0,"cost_eur":7200.0,"status":"Completed","description":"Generator winding inspection and oil change","next_maintenance_date":"2025-07-08","downtime_hours":4.0},
    {"maintenance_id":"MNT_008","substation_id":"SUB_013","maintenance_type":"Corrective","component":"Rotor Blades","scheduled_date":"2024-08-20","completed_date":"2024-08-23","technician_id":"TECH_09","duration_hours":24.0,"cost_eur":18000.0,"status":"Completed","description":"Blade crack repair after storm damage","next_maintenance_date":"2025-08-20","downtime_hours":24.0},
    {"maintenance_id":"MNT_009","substation_id":"SUB_016","maintenance_type":"Preventive","component":"Nacelle","scheduled_date":"2024-09-14","completed_date":"2024-09-14","technician_id":"TECH_11","duration_hours":7.0,"cost_eur":4100.0,"status":"Completed","description":"Nacelle lubrication and yaw system calibration","next_maintenance_date":"2025-09-14","downtime_hours":2.0},
    {"maintenance_id":"MNT_010","substation_id":"SUB_017","maintenance_type":"Inspection","component":"Dam Infrastructure","scheduled_date":"2024-10-05","completed_date":"2024-10-06","technician_id":"TECH_04","duration_hours":16.0,"cost_eur":14000.0,"status":"Completed","description":"Structural integrity inspection of dam body and gates","next_maintenance_date":"2025-10-05","downtime_hours":0.0},
    {"maintenance_id":"MNT_011","substation_id":"SUB_019","maintenance_type":"Preventive","component":"Tracking System","scheduled_date":"2024-11-18","completed_date":"2024-11-18","technician_id":"TECH_06","duration_hours":4.0,"cost_eur":1900.0,"status":"Completed","description":"Solar tracker motor and sensor calibration","next_maintenance_date":"2025-05-18","downtime_hours":1.0},
    {"maintenance_id":"MNT_012","substation_id":"SUB_020","maintenance_type":"Preventive","component":"Transformer","scheduled_date":"2025-01-22","completed_date":None,"technician_id":"TECH_10","duration_hours":None,"cost_eur":None,"status":"Scheduled","description":"Transformer oil analysis and bushing inspection","next_maintenance_date":"2026-01-22","downtime_hours":None},
]

df_maintenance = pd.DataFrame(maintenance_data)
print(f"DIM_MAINTENANCE  |  {len(df_maintenance)} rows")
print(df_maintenance.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 10, Finished, Available, Finished, False)

DIM_MAINTENANCE  |  12 rows
maintenance_id substation_id maintenance_type          component scheduled_date completed_date technician_id  duration_hours  cost_eur    status                                           description next_maintenance_date  downtime_hours
       MNT_001       SUB_001       Preventive       Solar Panels     2024-02-10     2024-02-10       TECH_03             6.0    3200.0 Completed            Annual panel cleaning and efficiency check            2025-02-10             2.0
       MNT_002       SUB_002       Corrective            Gearbox     2024-03-05     2024-03-07       TECH_07            18.0   12500.0 Completed   Gearbox bearing replacement after vibration anomaly            2025-03-05            18.0
       MNT_003       SUB_003       Preventive    Tidal Converter     2024-01-20     2024-01-21       TECH_12            10.0    8900.0 Completed         Marine biofouling removal and seal inspection            2024-07-20            10.0
       MNT_004       SUB

In [4]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_TECHNICIAN  –  Workforce & Certification Reference
# Ontology role: Person/Agent who PERFORMS Maintenance on Equipment at Substation
# ─────────────────────────────────────────────────────────────────────────────
technicians_data = [
    {"technician_id":"TECH_01","full_name":"Carlos García Romero","specialization":"Hydroelectric Systems","certification_level":"Senior Engineer","years_experience":15,"base_location":"Toledo","certifications":"IEC 60041, CEI EN 61400","languages":"Spanish, English","hourly_rate_eur":85.0,"active":True,"emergency_contact":"+34-612-001-001"},
    {"technician_id":"TECH_02","full_name":"Ainhoa Etxeberria Zubiaurre","specialization":"Hydroelectric Systems","certification_level":"Senior Engineer","years_experience":12,"base_location":"Bilbao","certifications":"IEC 60041, CEWEP","languages":"Spanish, Basque, French","hourly_rate_eur":82.0,"active":True,"emergency_contact":"+34-612-002-002"},
    {"technician_id":"TECH_03","full_name":"María Fernández López","specialization":"Solar PV Systems","certification_level":"Field Engineer","years_experience":7,"base_location":"Madrid","certifications":"IEC 62446, NABCEP PV Associate","languages":"Spanish, English","hourly_rate_eur":65.0,"active":True,"emergency_contact":"+34-612-003-003"},
    {"technician_id":"TECH_04","full_name":"Jorge Navarro Blanco","specialization":"Civil & Structural","certification_level":"Senior Engineer","years_experience":20,"base_location":"Cáceres","certifications":"Eurocode 2, ICOLD Guidelines","languages":"Spanish, Portuguese","hourly_rate_eur":90.0,"active":True,"emergency_contact":"+34-612-004-004"},
    {"technician_id":"TECH_05","full_name":"Laura Castillo Vega","specialization":"Power Electronics","certification_level":"Field Engineer","years_experience":9,"base_location":"Almeria","certifications":"IEC 62109, ISO 9001","languages":"Spanish, English, German","hourly_rate_eur":70.0,"active":True,"emergency_contact":"+34-612-005-005"},
    {"technician_id":"TECH_06","full_name":"Rubén Morales Sanz","specialization":"Solar PV Systems","certification_level":"Technician","years_experience":5,"base_location":"Cordoba","certifications":"NABCEP PV Associate, IEC 62446","languages":"Spanish","hourly_rate_eur":52.0,"active":True,"emergency_contact":"+34-612-006-006"},
    {"technician_id":"TECH_07","full_name":"Iñigo Urresti Arizmendi","specialization":"Wind Turbine Systems","certification_level":"Senior Engineer","years_experience":14,"base_location":"Bilbao","certifications":"GWO BST, IEC 61400-1","languages":"Spanish, Basque, English","hourly_rate_eur":88.0,"active":True,"emergency_contact":"+34-612-007-007"},
    {"technician_id":"TECH_08","full_name":"Carmen Delgado Ruiz","specialization":"Solar PV Systems","certification_level":"Field Engineer","years_experience":6,"base_location":"Malaga","certifications":"IEC 62446, CIEMAT Certification","languages":"Spanish, English","hourly_rate_eur":62.0,"active":True,"emergency_contact":"+34-612-008-008"},
    {"technician_id":"TECH_09","full_name":"Alejandro Torres Prado","specialization":"Wind Turbine Systems","certification_level":"Field Engineer","years_experience":10,"base_location":"Oviedo","certifications":"GWO BST, GWO BOSIET, IEC 61400-1","languages":"Spanish, English","hourly_rate_eur":75.0,"active":True,"emergency_contact":"+34-612-009-009"},
    {"technician_id":"TECH_10","full_name":"Pilar Santos Aguilar","specialization":"Power Transformers","certification_level":"Field Engineer","years_experience":8,"base_location":"León","certifications":"IEC 60076, IEEE C57","languages":"Spanish, English","hourly_rate_eur":68.0,"active":True,"emergency_contact":"+34-612-010-010"},
    {"technician_id":"TECH_11","full_name":"David Herrera Montes","specialization":"Wind Turbine Systems","certification_level":"Technician","years_experience":4,"base_location":"Pamplona","certifications":"GWO BST","languages":"Spanish, English","hourly_rate_eur":55.0,"active":True,"emergency_contact":"+34-612-011-011"},
    {"technician_id":"TECH_12","full_name":"Eva Serrano Costa","specialization":"Marine Energy Systems","certification_level":"Senior Engineer","years_experience":11,"base_location":"Valencia","certifications":"EMEC Guidelines, IEC TC114","languages":"Spanish, English, French","hourly_rate_eur":92.0,"active":True,"emergency_contact":"+34-612-012-012"},
]

df_technicians = pd.DataFrame(technicians_data)
print(f"DIM_TECHNICIAN  |  {len(df_technicians)} rows")
print(df_technicians.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 11, Finished, Available, Finished, False)

DIM_TECHNICIAN  |  12 rows
technician_id                   full_name        specialization certification_level  years_experience base_location                   certifications                languages  hourly_rate_eur  active emergency_contact
      TECH_01        Carlos García Romero Hydroelectric Systems     Senior Engineer                15        Toledo          IEC 60041, CEI EN 61400         Spanish, English             85.0    True   +34-612-001-001
      TECH_02 Ainhoa Etxeberria Zubiaurre Hydroelectric Systems     Senior Engineer                12        Bilbao                 IEC 60041, CEWEP  Spanish, Basque, French             82.0    True   +34-612-002-002
      TECH_03       María Fernández López      Solar PV Systems      Field Engineer                 7        Madrid   IEC 62446, NABCEP PV Associate         Spanish, English             65.0    True   +34-612-003-003
      TECH_04        Jorge Navarro Blanco    Civil & Structural     Senior Engineer                20    

In [5]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_WEATHER_STATION  –  Weather & Climate Reference Data
# Ontology role: WeatherStation OBSERVES Environment AT GeoLocation NEAR Substation
#   → enables correlating ambient_temperature_c / efficiency with climate patterns
# ─────────────────────────────────────────────────────────────────────────────
weather_stations_data = [
    {"station_id":"WS_001","station_name":"Madrid-Barajas AEMET","substation_id_nearest":"SUB_001","latitude":40.4719,"longitude":-3.5567,"altitude_m":609,"climate_zone":"BSk","avg_annual_sunshine_hours":2769,"avg_annual_wind_speed_ms":4.2,"avg_annual_precipitation_mm":436,"avg_annual_temp_c":15.1,"data_source":"AEMET"},
    {"station_id":"WS_002","station_name":"Barcelona El Prat AEMET","substation_id_nearest":"SUB_002","latitude":41.2974,"longitude":2.0833,"altitude_m":4,"climate_zone":"Csa","avg_annual_sunshine_hours":2524,"avg_annual_wind_speed_ms":5.8,"avg_annual_precipitation_mm":600,"avg_annual_temp_c":17.2,"data_source":"AEMET"},
    {"station_id":"WS_003","station_name":"Valencia Airport AEMET","substation_id_nearest":"SUB_003","latitude":39.4893,"longitude":-0.4812,"altitude_m":69,"climate_zone":"BSh","avg_annual_sunshine_hours":2696,"avg_annual_wind_speed_ms":3.9,"avg_annual_precipitation_mm":454,"avg_annual_temp_c":18.5,"data_source":"AEMET"},
    {"station_id":"WS_004","station_name":"Almeria Airport AEMET","substation_id_nearest":"SUB_007","latitude":36.8439,"longitude":-2.3645,"altitude_m":21,"climate_zone":"BWh","avg_annual_sunshine_hours":3029,"avg_annual_wind_speed_ms":4.6,"avg_annual_precipitation_mm":196,"avg_annual_temp_c":19.0,"data_source":"AEMET"},
    {"station_id":"WS_005","station_name":"Seville San Pablo AEMET","substation_id_nearest":"SUB_005","latitude":37.4180,"longitude":-5.8931,"altitude_m":34,"climate_zone":"Csa","avg_annual_sunshine_hours":3044,"avg_annual_wind_speed_ms":3.5,"avg_annual_precipitation_mm":534,"avg_annual_temp_c":19.3,"data_source":"AEMET"},
    {"station_id":"WS_006","station_name":"Bilbao Airport AEMET","substation_id_nearest":"SUB_006","latitude":43.3010,"longitude":-2.9106,"altitude_m":42,"climate_zone":"Cfb","avg_annual_sunshine_hours":1661,"avg_annual_wind_speed_ms":6.3,"avg_annual_precipitation_mm":1158,"avg_annual_temp_c":14.3,"data_source":"AEMET"},
    {"station_id":"WS_007","station_name":"A Coruña AEMET","substation_id_nearest":"SUB_008","latitude":43.3660,"longitude":-8.4225,"altitude_m":67,"climate_zone":"Cfb","avg_annual_sunshine_hours":1844,"avg_annual_wind_speed_ms":7.1,"avg_annual_precipitation_mm":1008,"avg_annual_temp_c":14.5,"data_source":"AEMET"},
    {"station_id":"WS_008","station_name":"Zaragoza Airport AEMET","substation_id_nearest":"SUB_004","latitude":41.6622,"longitude":-1.0456,"altitude_m":263,"climate_zone":"BSk","avg_annual_sunshine_hours":2863,"avg_annual_wind_speed_ms":6.9,"avg_annual_precipitation_mm":322,"avg_annual_temp_c":15.5,"data_source":"AEMET"},
]

df_weather_stations = pd.DataFrame(weather_stations_data)
print(f"DIM_WEATHER_STATION  |  {len(df_weather_stations)} rows")
print(df_weather_stations.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 12, Finished, Available, Finished, False)

DIM_WEATHER_STATION  |  8 rows
station_id            station_name substation_id_nearest  latitude  longitude  altitude_m climate_zone  avg_annual_sunshine_hours  avg_annual_wind_speed_ms  avg_annual_precipitation_mm  avg_annual_temp_c data_source
    WS_001    Madrid-Barajas AEMET               SUB_001   40.4719    -3.5567         609          BSk                       2769                       4.2                          436               15.1       AEMET
    WS_002 Barcelona El Prat AEMET               SUB_002   41.2974     2.0833           4          Csa                       2524                       5.8                          600               17.2       AEMET
    WS_003  Valencia Airport AEMET               SUB_003   39.4893    -0.4812          69          BSh                       2696                       3.9                          454               18.5       AEMET
    WS_004   Almeria Airport AEMET               SUB_007   36.8439    -2.3645          21          BWh   

In [6]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_ENERGY_TARIFF  –  Pricing & Market Segment Reference
# Ontology role: EnergyTariff GOVERNS how a Substation is COMPENSATED
#   Links energy_type → market mechanism (Regulated / Merchant / FiT / PPA)
# ─────────────────────────────────────────────────────────────────────────────
tariff_data = [
    {"tariff_id":"TARIFF_001","tariff_name":"PVPC Off-Peak","energy_type":"Solar","market_segment":"Regulated","time_band":"Off-Peak","valid_from":"2024-01-01","valid_to":"2024-12-31","price_eur_per_kwh":0.0823,"vat_rate_pct":10.0,"grid_access_charge_eur_kwh":0.0142,"regulatory_body":"CNMC","applies_to_region":"All Spain","currency":"EUR"},
    {"tariff_id":"TARIFF_002","tariff_name":"PVPC Peak","energy_type":"Solar","market_segment":"Regulated","time_band":"Peak","valid_from":"2024-01-01","valid_to":"2024-12-31","price_eur_per_kwh":0.1456,"vat_rate_pct":10.0,"grid_access_charge_eur_kwh":0.0142,"regulatory_body":"CNMC","applies_to_region":"All Spain","currency":"EUR"},
    {"tariff_id":"TARIFF_003","tariff_name":"Wind Merchant Day-Ahead","energy_type":"Wind","market_segment":"Merchant","time_band":"Day-Ahead","valid_from":"2024-01-01","valid_to":"2024-12-31","price_eur_per_kwh":0.0967,"vat_rate_pct":10.0,"grid_access_charge_eur_kwh":0.0135,"regulatory_body":"OMIE","applies_to_region":"Iberian Peninsula","currency":"EUR"},
    {"tariff_id":"TARIFF_004","tariff_name":"Hydroelectric Baseload","energy_type":"Hydroelectric","market_segment":"Merchant","time_band":"Baseload","valid_from":"2024-01-01","valid_to":"2024-12-31","price_eur_per_kwh":0.0745,"vat_rate_pct":10.0,"grid_access_charge_eur_kwh":0.0128,"regulatory_body":"OMIE","applies_to_region":"Iberian Peninsula","currency":"EUR"},
    {"tariff_id":"TARIFF_005","tariff_name":"Marine FiT Contract","energy_type":"Marine","market_segment":"Feed-in Tariff","time_band":"All Hours","valid_from":"2023-01-01","valid_to":"2030-12-31","price_eur_per_kwh":0.1890,"vat_rate_pct":10.0,"grid_access_charge_eur_kwh":0.0,"regulatory_body":"MITECO","applies_to_region":"Coastal Regions","currency":"EUR"},
    {"tariff_id":"TARIFF_006","tariff_name":"Solar PPA Corporate","energy_type":"Solar","market_segment":"PPA","time_band":"All Hours","valid_from":"2023-06-01","valid_to":"2033-05-31","price_eur_per_kwh":0.0680,"vat_rate_pct":21.0,"grid_access_charge_eur_kwh":0.0,"regulatory_body":"Private Contract","applies_to_region":"All Spain","currency":"EUR"},
]

df_tariffs = pd.DataFrame(tariff_data)
print(f"DIM_ENERGY_TARIFF  |  {len(df_tariffs)} rows")
print(df_tariffs.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 13, Finished, Available, Finished, False)

DIM_ENERGY_TARIFF  |  6 rows
 tariff_id             tariff_name   energy_type market_segment time_band valid_from   valid_to  price_eur_per_kwh  vat_rate_pct  grid_access_charge_eur_kwh  regulatory_body applies_to_region currency
TARIFF_001           PVPC Off-Peak         Solar      Regulated  Off-Peak 2024-01-01 2024-12-31             0.0823          10.0                      0.0142             CNMC         All Spain      EUR
TARIFF_002               PVPC Peak         Solar      Regulated      Peak 2024-01-01 2024-12-31             0.1456          10.0                      0.0142             CNMC         All Spain      EUR
TARIFF_003 Wind Merchant Day-Ahead          Wind       Merchant Day-Ahead 2024-01-01 2024-12-31             0.0967          10.0                      0.0135             OMIE Iberian Peninsula      EUR
TARIFF_004  Hydroelectric Baseload Hydroelectric       Merchant  Baseload 2024-01-01 2024-12-31             0.0745          10.0                      0.0128           

In [7]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_EQUIPMENT  –  Physical Asset & Component Registry
# Ontology roles:
#   Equipment  IS_PART_OF  Substation
#   Equipment  HAS_MANUFACTURER  Manufacturer (string → can be expanded to entity)
#   Equipment  IS_MAINTAINED_VIA  Maintenance (join on component name + substation)
# ─────────────────────────────────────────────────────────────────────────────
equipment_data = [
    {"equipment_id":"EQ_001","substation_id":"SUB_001","component_name":"Solar Panel Array","manufacturer":"LONGi Green Energy","model":"Hi-MO 6 LR5-72HTH-570M","serial_number_prefix":"LGE-SUB001","installation_date":"2020-05-15","warranty_expiry":"2045-05-15","rated_power_kw":0.57,"quantity":150000,"efficiency_rating_pct":22.3,"expected_lifetime_years":25,"replacement_cost_eur":4500000.0,"last_inspection":"2024-02-10","condition":"Good"},
    {"equipment_id":"EQ_002","substation_id":"SUB_001","component_name":"String Inverter","manufacturer":"Huawei","model":"SUN2000-100KTL-M2","serial_number_prefix":"HW-INV-SUB001","installation_date":"2020-05-15","warranty_expiry":"2030-05-15","rated_power_kw":100.0,"quantity":500,"efficiency_rating_pct":98.8,"expected_lifetime_years":10,"replacement_cost_eur":750000.0,"last_inspection":"2024-02-10","condition":"Good"},
    {"equipment_id":"EQ_003","substation_id":"SUB_002","component_name":"Wind Turbine","manufacturer":"Vestas","model":"V150-4.5 MW","serial_number_prefix":"VST-SUB002","installation_date":"2019-03-22","warranty_expiry":"2029-03-22","rated_power_kw":4500.0,"quantity":40,"efficiency_rating_pct":96.5,"expected_lifetime_years":25,"replacement_cost_eur":9200000.0,"last_inspection":"2024-03-07","condition":"Repaired"},
    {"equipment_id":"EQ_004","substation_id":"SUB_003","component_name":"Wave Energy Converter","manufacturer":"Orca Wave Energy","model":"OWE-2000","serial_number_prefix":"OWE-SUB003","installation_date":"2021-09-10","warranty_expiry":"2031-09-10","rated_power_kw":2000.0,"quantity":15,"efficiency_rating_pct":88.2,"expected_lifetime_years":20,"replacement_cost_eur":2100000.0,"last_inspection":"2024-01-21","condition":"Good"},
    {"equipment_id":"EQ_005","substation_id":"SUB_005","component_name":"Francis Turbine","manufacturer":"Voith","model":"VH-FT-2500","serial_number_prefix":"VH-SUB005","installation_date":"2017-06-12","warranty_expiry":"2027-06-12","rated_power_kw":25000.0,"quantity":4,"efficiency_rating_pct":94.1,"expected_lifetime_years":40,"replacement_cost_eur":3800000.0,"last_inspection":"2024-04-15","condition":"Good"},
    {"equipment_id":"EQ_006","substation_id":"SUB_007","component_name":"Solar Panel Array","manufacturer":"JA Solar","model":"JAM72D40-570/MB","serial_number_prefix":"JAS-SUB007","installation_date":"2020-01-20","warranty_expiry":"2045-01-20","rated_power_kw":0.57,"quantity":270000,"efficiency_rating_pct":21.8,"expected_lifetime_years":25,"replacement_cost_eur":8100000.0,"last_inspection":"2024-05-02","condition":"Fair"},
    {"equipment_id":"EQ_007","substation_id":"SUB_007","component_name":"Central Inverter","manufacturer":"SMA Solar","model":"SUNNY CENTRAL 4200 UP","serial_number_prefix":"SMA-INV-SUB007","installation_date":"2020-01-20","warranty_expiry":"2030-01-20","rated_power_kw":4200.0,"quantity":22,"efficiency_rating_pct":99.0,"expected_lifetime_years":10,"replacement_cost_eur":660000.0,"last_inspection":"2024-05-02","condition":"Replaced"},
    {"equipment_id":"EQ_008","substation_id":"SUB_016","component_name":"Wind Turbine","manufacturer":"Siemens Gamesa","model":"SG 5.0-145","serial_number_prefix":"SG-SUB016","installation_date":"2019-12-15","warranty_expiry":"2029-12-15","rated_power_kw":5000.0,"quantity":41,"efficiency_rating_pct":97.2,"expected_lifetime_years":25,"replacement_cost_eur":10250000.0,"last_inspection":"2024-09-14","condition":"Good"},
    {"equipment_id":"EQ_009","substation_id":"SUB_017","component_name":"Pelton Turbine","manufacturer":"GE Renewable Energy","model":"GE-PT-3000","serial_number_prefix":"GE-SUB017","installation_date":"2018-08-30","warranty_expiry":"2028-08-30","rated_power_kw":23750.0,"quantity":4,"efficiency_rating_pct":92.8,"expected_lifetime_years":40,"replacement_cost_eur":4200000.0,"last_inspection":"2024-10-06","condition":"Good"},
    {"equipment_id":"EQ_010","substation_id":"SUB_020","component_name":"Power Transformer","manufacturer":"ABB","model":"TrafoStar ONAF 80MVA","serial_number_prefix":"ABB-TF-SUB020","installation_date":"2020-05-22","warranty_expiry":"2035-05-22","rated_power_kw":80000.0,"quantity":1,"efficiency_rating_pct":99.4,"expected_lifetime_years":30,"replacement_cost_eur":1800000.0,"last_inspection":None,"condition":"Scheduled Inspection"},
]

df_equipment = pd.DataFrame(equipment_data)
print(f"DIM_EQUIPMENT  |  {len(df_equipment)} rows")
print(df_equipment.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 14, Finished, Available, Finished, False)

DIM_EQUIPMENT  |  10 rows
equipment_id substation_id        component_name        manufacturer                  model serial_number_prefix installation_date warranty_expiry  rated_power_kw  quantity  efficiency_rating_pct  expected_lifetime_years  replacement_cost_eur last_inspection            condition
      EQ_001       SUB_001     Solar Panel Array  LONGi Green Energy Hi-MO 6 LR5-72HTH-570M           LGE-SUB001        2020-05-15      2045-05-15            0.57    150000                   22.3                       25             4500000.0      2024-02-10                 Good
      EQ_002       SUB_001       String Inverter              Huawei      SUN2000-100KTL-M2        HW-INV-SUB001        2020-05-15      2030-05-15          100.00       500                   98.8                       10              750000.0      2024-02-10                 Good
      EQ_003       SUB_002          Wind Turbine              Vestas            V150-4.5 MW           VST-SUB002        2019-03-22    

In [8]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_REGULATION  –  Regulatory & Compliance Framework
# Ontology role: Regulation GOVERNS Substation operations (via bridge table).
#   Represents the legal / technical / policy context for energy generation.
# ─────────────────────────────────────────────────────────────────────────────
regulation_data = [
    {"regulation_id":"REG_001","regulation_name":"Royal Decree 244/2019","short_name":"RD244/2019","type":"Feed-in Tariff / Self-consumption","jurisdiction":"Spain","regulatory_body":"MITECO","effective_date":"2019-04-06","review_date":"2024-04-06","applies_to_energy_type":"Solar, Wind, Marine, Hydroelectric","max_capacity_threshold_mw":100.0,"compliance_penalty_eur":600000.0,"url":"https://www.boe.es/eli/es/rd/2019/04/05/244","status":"Active","notes":"Self-consumption without surplus regulation"},
    {"regulation_id":"REG_002","regulation_name":"Law 24/2013 - Electric Power Sector","short_name":"LSE 24/2013","type":"Market Regulation","jurisdiction":"Spain","regulatory_body":"CNMC","effective_date":"2013-12-27","review_date":"2023-12-27","applies_to_energy_type":"Solar, Wind, Marine, Hydroelectric","max_capacity_threshold_mw":None,"compliance_penalty_eur":1500000.0,"url":"https://www.boe.es/eli/es/l/2013/12/26/24","status":"Active","notes":"General electricity sector framework"},
    {"regulation_id":"REG_003","regulation_name":"EU Renewable Energy Directive II","short_name":"RED II (2018/2001)","type":"EU Directive","jurisdiction":"European Union","regulatory_body":"European Commission","effective_date":"2018-12-21","review_date":"2023-12-21","applies_to_energy_type":"Solar, Wind, Marine, Hydroelectric","max_capacity_threshold_mw":None,"compliance_penalty_eur":None,"url":"https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32018L2001","status":"Active","notes":"32% renewable target by 2030"},
    {"regulation_id":"REG_004","regulation_name":"Royal Decree 1183/2020","short_name":"RD1183/2020","type":"Grid Access","jurisdiction":"Spain","regulatory_body":"REE (Red Eléctrica de España)","effective_date":"2020-12-30","review_date":"2025-12-30","applies_to_energy_type":"Solar, Wind, Marine, Hydroelectric","max_capacity_threshold_mw":None,"compliance_penalty_eur":300000.0,"url":"https://www.boe.es/eli/es/rd/2020/12/29/1183","status":"Active","notes":"Grid access and connection permits"},
    {"regulation_id":"REG_005","regulation_name":"IEC 61400 Series - Wind Turbines","short_name":"IEC 61400","type":"Technical Standard","jurisdiction":"International","regulatory_body":"IEC TC88","effective_date":"2005-08-01","review_date":"2023-08-01","applies_to_energy_type":"Wind","max_capacity_threshold_mw":None,"compliance_penalty_eur":None,"url":"https://www.iec.ch/dyn/www/f?p=103:38:0::::FSP_ORG_ID:1282","status":"Active","notes":"Design requirements for wind turbines"},
    {"regulation_id":"REG_006","regulation_name":"Spanish National Energy and Climate Plan","short_name":"PNIEC 2021-2030","type":"National Policy","jurisdiction":"Spain","regulatory_body":"MITECO","effective_date":"2021-01-01","review_date":"2030-12-31","applies_to_energy_type":"Solar, Wind, Marine, Hydroelectric","max_capacity_threshold_mw":None,"compliance_penalty_eur":None,"url":"https://energia.gob.es/es-ES/Paginas/pniec.aspx","status":"Active","notes":"74% renewables in electricity by 2030 target"},
]

df_regulations = pd.DataFrame(regulation_data)
print(f"DIM_REGULATION  |  {len(df_regulations)} rows")
print(df_regulations.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 15, Finished, Available, Finished, False)

DIM_REGULATION  |  6 rows
regulation_id                          regulation_name         short_name                              type   jurisdiction               regulatory_body effective_date review_date             applies_to_energy_type  max_capacity_threshold_mw  compliance_penalty_eur                                                                  url status                                        notes
      REG_001                    Royal Decree 244/2019         RD244/2019 Feed-in Tariff / Self-consumption          Spain                        MITECO     2019-04-06  2024-04-06 Solar, Wind, Marine, Hydroelectric                      100.0                600000.0                          https://www.boe.es/eli/es/rd/2019/04/05/244 Active  Self-consumption without surplus regulation
      REG_002      Law 24/2013 - Electric Power Sector        LSE 24/2013                 Market Regulation          Spain                          CNMC     2013-12-27  2023-12-27 Solar, Wind, Marine,

In [9]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_GRID_OPERATOR  –  Grid Operators & Market Participants
# Ontology role: GridOperator MANAGES the Grid to which Substation CONNECTS_TO.
#   TSO vs DSO vs Market Operator are sub-classes of Organisation.
# ─────────────────────────────────────────────────────────────────────────────
grid_operators_data = [
    {"operator_id":"OPR_001","operator_name":"Red Eléctrica de España (REE)","operator_type":"Transmission System Operator","jurisdiction":"Spain","voltage_level_kv":"400 / 220 / 132","headquarters":"Alcobendas, Madrid","contact_email":"operacion@ree.es","emergency_phone":"+34-91-702-1000","grid_code":"REE-P.O.","owned_by":"SEPI (State)","annual_capacity_gw":105.0,"website":"https://www.ree.es","certifications":"ISO 9001, ISO 50001"},
    {"operator_id":"OPR_002","operator_name":"Endesa Distribución","operator_type":"Distribution System Operator","jurisdiction":"Andalusia, Catalonia, Aragon, Balearics, Canaries","voltage_level_kv":"132 / 66 / 20 / 15 / 13.2","headquarters":"Madrid","contact_email":"clientes@endesa.es","emergency_phone":"+34-800-760-333","grid_code":"Endesa DSO Code","owned_by":"Enel Group","annual_capacity_gw":35.0,"website":"https://www.endesa.com","certifications":"ISO 9001, ISO 14001"},
    {"operator_id":"OPR_003","operator_name":"Iberdrola Distribución","operator_type":"Distribution System Operator","jurisdiction":"Basque Country, Castilla y León, Navarre, Galicia","voltage_level_kv":"132 / 66 / 20 / 13.2","headquarters":"Bilbao","contact_email":"distribucion@iberdrola.es","emergency_phone":"+34-900-100-190","grid_code":"Iberdrola DSO Code","owned_by":"Iberdrola S.A.","annual_capacity_gw":32.0,"website":"https://www.iberdrola.es","certifications":"ISO 9001, ISO 45001"},
    {"operator_id":"OPR_004","operator_name":"OMIE - Iberian Energy Market Operator","operator_type":"Market Operator","jurisdiction":"Iberian Peninsula","voltage_level_kv":None,"headquarters":"Madrid","contact_email":"omie@omie.es","emergency_phone":"+34-91-515-3100","grid_code":"OMIE Market Rules","owned_by":"CNMC / ERSE","annual_capacity_gw":None,"website":"https://www.omie.es","certifications":"ISO 27001"},
]

df_grid_operators = pd.DataFrame(grid_operators_data)
print(f"DIM_GRID_OPERATOR  |  {len(df_grid_operators)} rows")
print(df_grid_operators.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 16, Finished, Available, Finished, False)

DIM_GRID_OPERATOR  |  4 rows
operator_id                         operator_name                operator_type                                      jurisdiction          voltage_level_kv       headquarters             contact_email emergency_phone          grid_code       owned_by  annual_capacity_gw                  website      certifications
    OPR_001         Red Eléctrica de España (REE) Transmission System Operator                                             Spain           400 / 220 / 132 Alcobendas, Madrid          operacion@ree.es +34-91-702-1000           REE-P.O.   SEPI (State)               105.0       https://www.ree.es ISO 9001, ISO 50001
    OPR_002                   Endesa Distribución Distribution System Operator Andalusia, Catalonia, Aragon, Balearics, Canaries 132 / 66 / 20 / 15 / 13.2             Madrid        clientes@endesa.es +34-800-760-333    Endesa DSO Code     Enel Group                35.0   https://www.endesa.com ISO 9001, ISO 14001
    OPR_003               

In [10]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# DIM_ALERT_TYPE  –  Alarm Taxonomy & SLA Reference
# Ontology role: AlertType CLASSIFIES an Anomaly detected in fact_energy_reading.
#   Bridges the real-time is_anomaly flag to a structured taxonomy with
#   severity, escalation path, required response SLA, and auto-shutoff policy.
# ─────────────────────────────────────────────────────────────────────────────
alert_types_data = [
    {"alert_type_id":"ALT_001","alert_name":"Power Spike","category":"Electrical Anomaly","severity":"Critical","energy_type_applicable":"Solar, Wind","threshold_condition":"generated_kwh > 2.5x baseline","auto_shutoff":True,"response_time_minutes":5,"escalation_level":"L3 - Plant Manager","notification_channel":"SMS, Email, SCADA","requires_inspection":True,"description":"Sudden surge in power output beyond safe operating thresholds"},
    {"alert_type_id":"ALT_002","alert_name":"Power Drop","category":"Electrical Anomaly","severity":"High","energy_type_applicable":"Solar, Wind, Marine, Hydroelectric","threshold_condition":"generated_kwh < 0.15x baseline","auto_shutoff":False,"response_time_minutes":15,"escalation_level":"L2 - Operations Supervisor","notification_channel":"Email, SCADA","requires_inspection":True,"description":"Unexpected drop in energy output indicating possible equipment failure"},
    {"alert_type_id":"ALT_003","alert_name":"Overtemperature","category":"Thermal","severity":"High","energy_type_applicable":"Solar","threshold_condition":"ambient_temperature_c > 45 combined with efficiency drop","auto_shutoff":False,"response_time_minutes":20,"escalation_level":"L2 - Operations Supervisor","notification_channel":"Email, SCADA","requires_inspection":False,"description":"Panel operating temperature exceeds safe efficiency range"},
    {"alert_type_id":"ALT_004","alert_name":"Grid Voltage Deviation","category":"Grid Quality","severity":"Critical","energy_type_applicable":"Solar, Wind, Marine, Hydroelectric","threshold_condition":"grid_voltage_kv outside +-10% of nominal","auto_shutoff":True,"response_time_minutes":2,"escalation_level":"L3 - Plant Manager + Grid Operator","notification_channel":"SMS, Email, SCADA, Phone","requires_inspection":True,"description":"Grid voltage deviates outside permissible limits"},
    {"alert_type_id":"ALT_005","alert_name":"Low Efficiency Warning","category":"Performance","severity":"Medium","energy_type_applicable":"Solar, Wind, Marine, Hydroelectric","threshold_condition":"efficiency_percent < 80","auto_shutoff":False,"response_time_minutes":60,"escalation_level":"L1 - Technician","notification_channel":"Email","requires_inspection":False,"description":"Efficiency below acceptable threshold, scheduled review required"},
    {"alert_type_id":"ALT_006","alert_name":"Communication Loss","category":"Connectivity","severity":"Medium","energy_type_applicable":"Solar, Wind, Marine, Hydroelectric","threshold_condition":"No telemetry data for > 5 minutes","auto_shutoff":False,"response_time_minutes":10,"escalation_level":"L1 - Technician","notification_channel":"SMS, SCADA","requires_inspection":False,"description":"Sensor or connectivity failure, data stream interrupted"},
    {"alert_type_id":"ALT_007","alert_name":"Storm Warning","category":"Environmental","severity":"High","energy_type_applicable":"Wind, Marine","threshold_condition":"Wind speed forecast > 25 m/s within 6 hours","auto_shutoff":True,"response_time_minutes":30,"escalation_level":"L2 - Operations Supervisor","notification_channel":"SMS, Email, SCADA","requires_inspection":False,"description":"Severe weather forecast triggers protective shutdown procedure"},
    {"alert_type_id":"ALT_008","alert_name":"Maintenance Overdue","category":"Compliance","severity":"Low","energy_type_applicable":"Solar, Wind, Marine, Hydroelectric","threshold_condition":"next_maintenance_date exceeded by 7 days","auto_shutoff":False,"response_time_minutes":1440,"escalation_level":"L1 - Technician","notification_channel":"Email","requires_inspection":True,"description":"Scheduled maintenance window has passed without completion"},
]

df_alert_types = pd.DataFrame(alert_types_data)
print(f"DIM_ALERT_TYPE  |  {len(df_alert_types)} rows")
print(df_alert_types.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 17, Finished, Available, Finished, False)

DIM_ALERT_TYPE  |  8 rows
alert_type_id             alert_name           category severity             energy_type_applicable                                      threshold_condition  auto_shutoff  response_time_minutes                   escalation_level     notification_channel  requires_inspection                                                            description
      ALT_001            Power Spike Electrical Anomaly Critical                        Solar, Wind                            generated_kwh > 2.5x baseline          True                      5                 L3 - Plant Manager        SMS, Email, SCADA                 True          Sudden surge in power output beyond safe operating thresholds
      ALT_002             Power Drop Electrical Anomaly     High Solar, Wind, Marine, Hydroelectric                           generated_kwh < 0.15x baseline         False                     15         L2 - Operations Supervisor             Email, SCADA                 True Unexpec

In [11]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# BRIDGE_SUBSTATION_REGULATION  –  M:M Compliance Mapping
# Ontology role: Resolves the many-to-many between Substation and Regulation.
#   Each row = a COMPLIANCE ASSERTION (who audited, when, current status).
# ─────────────────────────────────────────────────────────────────────────────
bridge_data = [
    {"substation_id":"SUB_001","regulation_id":"REG_001","compliance_status":"Compliant","last_audit":"2024-03-01","next_audit":"2025-03-01","auditor":"AENOR"},
    {"substation_id":"SUB_001","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-03-01","next_audit":"2025-03-01","auditor":"CNMC"},
    {"substation_id":"SUB_001","regulation_id":"REG_004","compliance_status":"Compliant","last_audit":"2024-03-01","next_audit":"2025-03-01","auditor":"REE"},
    {"substation_id":"SUB_002","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-04-15","next_audit":"2025-04-15","auditor":"CNMC"},
    {"substation_id":"SUB_002","regulation_id":"REG_005","compliance_status":"Compliant","last_audit":"2024-04-15","next_audit":"2025-04-15","auditor":"TÜV SÜD"},
    {"substation_id":"SUB_003","regulation_id":"REG_003","compliance_status":"Compliant","last_audit":"2024-02-28","next_audit":"2025-02-28","auditor":"Bureau Veritas"},
    {"substation_id":"SUB_003","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-02-28","next_audit":"2025-02-28","auditor":"CNMC"},
    {"substation_id":"SUB_005","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-06-20","next_audit":"2025-06-20","auditor":"CNMC"},
    {"substation_id":"SUB_005","regulation_id":"REG_006","compliance_status":"Compliant","last_audit":"2024-06-20","next_audit":"2025-06-20","auditor":"MITECO"},
    {"substation_id":"SUB_007","regulation_id":"REG_001","compliance_status":"Under Review","last_audit":"2024-05-10","next_audit":"2024-11-10","auditor":"AENOR"},
    {"substation_id":"SUB_007","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-05-10","next_audit":"2025-05-10","auditor":"CNMC"},
    {"substation_id":"SUB_011","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-07-12","next_audit":"2025-07-12","auditor":"CNMC"},
    {"substation_id":"SUB_016","regulation_id":"REG_005","compliance_status":"Compliant","last_audit":"2024-07-01","next_audit":"2025-07-01","auditor":"TÜV SÜD"},
    {"substation_id":"SUB_016","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-07-01","next_audit":"2025-07-01","auditor":"CNMC"},
    {"substation_id":"SUB_017","regulation_id":"REG_002","compliance_status":"Compliant","last_audit":"2024-08-05","next_audit":"2025-08-05","auditor":"CNMC"},
    {"substation_id":"SUB_017","regulation_id":"REG_006","compliance_status":"Compliant","last_audit":"2024-08-05","next_audit":"2025-08-05","auditor":"MITECO"},
]

df_bridge = pd.DataFrame(bridge_data)
print(f"BRIDGE_SUBSTATION_REGULATION  |  {len(df_bridge)} rows")
print(df_bridge.to_string(index=False))

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 18, Finished, Available, Finished, False)

BRIDGE_SUBSTATION_REGULATION  |  16 rows
substation_id regulation_id compliance_status last_audit next_audit        auditor
      SUB_001       REG_001         Compliant 2024-03-01 2025-03-01          AENOR
      SUB_001       REG_002         Compliant 2024-03-01 2025-03-01           CNMC
      SUB_001       REG_004         Compliant 2024-03-01 2025-03-01            REE
      SUB_002       REG_002         Compliant 2024-04-15 2025-04-15           CNMC
      SUB_002       REG_005         Compliant 2024-04-15 2025-04-15        TÜV SÜD
      SUB_003       REG_003         Compliant 2024-02-28 2025-02-28 Bureau Veritas
      SUB_003       REG_002         Compliant 2024-02-28 2025-02-28           CNMC
      SUB_005       REG_002         Compliant 2024-06-20 2025-06-20           CNMC
      SUB_005       REG_006         Compliant 2024-06-20 2025-06-20         MITECO
      SUB_007       REG_001      Under Review 2024-05-10 2024-11-10          AENOR
      SUB_007       REG_002         Compliant 

In [12]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL ENRICHMENT TABLES → Lakehouse Delta + Parquet
# Run after all dim cells above so DataFrames exist in session scope.
# ─────────────────────────────────────────────────────────────────────────────

tables_to_save = {
    "dim_maintenance":               df_maintenance,
    "dim_technician":                df_technicians,
    "dim_weather_station":           df_weather_stations,
    "dim_energy_tariff":             df_tariffs,
    "dim_equipment":                 df_equipment,
    "dim_regulation":                df_regulations,
    "dim_grid_operator":             df_grid_operators,
    "dim_alert_type":                df_alert_types,
    "bridge_substation_regulation":  df_bridge,
}

print("=" * 80)
print("SAVING ENRICHMENT TABLES TO LAKEHOUSE")
print("=" * 80)

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    for table_name, df_pandas in tables_to_save.items():
        df_safe = df_pandas.astype(object).where(pd.notnull(df_pandas), None)
        spark_df = spark.createDataFrame(df_safe)
        spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark_df.write.mode("overwrite").parquet(f"Files/dimensions/{table_name}.parquet")
        print(f"  ✅ {table_name:45s} | {len(df_pandas):>3} rows")

except Exception as e:
    print(f"⚠️  Lakehouse save skipped – attach this notebook to a Lakehouse first.")
    print(f"   Error: {e}")
    for table_name, df_pandas in tables_to_save.items():
        df_pandas.to_csv(f"{table_name}.csv", index=False)
        print(f"  📁 {table_name}.csv  ({len(df_pandas)} rows) – saved locally")

print()
print("=" * 80)
print("ONTOLOGY ENTITY MAP  –  ready for OWL / RDF / Property Graph export")
print("=" * 80)
rows = [
    ("fact_energy_reading",          "Fact / Event stream", "Eventstream → Lakehouse"),
    ("dim_substation",               "Core entity",         "20 substations across Spain"),
    ("dim_equipment",                "Physical asset",      f"{len(df_equipment)} components across substations"),
    ("dim_technician",               "Person / Agent",      f"{len(df_technicians)} certified technicians"),
    ("dim_maintenance",              "Event",               f"{len(df_maintenance)} maintenance records"),
    ("dim_weather_station",          "Observation context", f"{len(df_weather_stations)} AEMET stations"),
    ("dim_energy_tariff",            "Economic rule",       f"{len(df_tariffs)} tariff segments"),
    ("dim_grid_operator",            "Organisation",        f"{len(df_grid_operators)} TSO / DSO / Market ops"),
    ("dim_regulation",               "Legal constraint",    f"{len(df_regulations)} regulations & standards"),
    ("dim_alert_type",               "Classification",      f"{len(df_alert_types)} alert types with SLA"),
    ("bridge_substation_regulation", "M:M relationship",    f"{len(df_bridge)} compliance assertions"),
]
print(f"  {'Table':<45} {'Ontology Role':<25} Details")
print("  " + "-" * 90)
for t, r, d in rows:
    print(f"  {t:<45} {r:<25} {d}")
print()
print("✅ All tables persisted. Proceed to ontology modelling.")

StatementMeta(, 6844d18b-7dbc-41f8-9abf-2dbd6f9b0f73, 19, Finished, Available, Finished, False)

SAVING ENRICHMENT TABLES TO LAKEHOUSE
  ✅ dim_maintenance                               |  12 rows
  ✅ dim_technician                                |  12 rows
  ✅ dim_weather_station                           |   8 rows
  ✅ dim_energy_tariff                             |   6 rows
  ✅ dim_equipment                                 |  10 rows
  ✅ dim_regulation                                |   6 rows
  ✅ dim_grid_operator                             |   4 rows
  ✅ dim_alert_type                                |   8 rows
  ✅ bridge_substation_regulation                  |  16 rows

ONTOLOGY ENTITY MAP  –  ready for OWL / RDF / Property Graph export
  Table                                         Ontology Role             Details
  ------------------------------------------------------------------------------------------
  fact_energy_reading                           Fact / Event stream       Eventstream → Lakehouse
  dim_substation                                Core entity            